# Root segmentation: U-Net training

Trains a Residual Attention U-Net on 256 × 256 grayscale patches of petri dish scans, using a hybrid binary cross-entropy + focal Tversky loss.

- **Input:** prepared patches in `../data/patches/` (`train_images/`, `train_masks/`, `val_images/`, `val_masks/`).
- **Output:** `../models/unet_model_256px.h5`.
- **Best validation F1:** 0.881.

In [ ]:
# 1. LIBRARY IMPORTS
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt

# Deep Learning Framework (TensorFlow/Keras)
import tensorflow as tf
from tensorflow.keras import backend as K
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.layers import *
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

# Verify TensorFlow is using the GPU (Optional but recommended check)
print(f"TensorFlow Version: {tf.__version__}")
print(f"Devices available: {tf.config.list_physical_devices('GPU')}")

In [ ]:
# 2. GLOBAL CONFIGURATION & CONSTANTS
# CONFIG 
PATCH_SIZE = 256  # Input resolution for the U-Net (256x256 pixels)
BATCH_SIZE = 32   # Number of samples processed per gradient update
PATCH_DIR = "../data/patches"  # train_images/, train_masks/, val_images/, val_masks/ (see README)

In [ ]:
# 3. CUSTOM METRIC: F1 SCORE
def f1_metric(y_true, y_pred):
    """
    Calculates the F1 Score (Harmonic mean of Precision and Recall).
    Useful for imbalanced datasets where accuracy can be misleading.
    """
    def recall_m(y_true, y_pred):
        TP = K.sum(K.round(K.clip(y_true * y_pred, 0, 1)))
        Positives = K.sum(K.round(K.clip(y_true, 0, 1)))
        recall = TP / (Positives + K.epsilon())
        return recall
    
    def precision_m(y_true, y_pred):
        TP = K.sum(K.round(K.clip(y_true * y_pred, 0, 1)))
        Pred_Positives = K.sum(K.round(K.clip(y_pred, 0, 1)))
        precision = TP / (Pred_Positives + K.epsilon())
        return precision
    
    precision, recall = precision_m(y_true, y_pred), recall_m(y_true, y_pred)
    
    return 2*((precision*recall)/(precision+recall+K.epsilon()))

In [ ]:
# 4. CUSTOM LOSS FUNCTIONS
# Challenge: Roots are thin (imbalance) and faint (hard examples).
# Solution: A Hybrid Loss combining stability (BCE) with sensitivity (Tversky).
def focal_tversky_loss(y_true, y_pred):
    """
    Optimized for Class Imbalance.
    - alpha=0.3, beta=0.7: Penalizes missed roots (False Negatives) more than noise.
    - gamma=4/3: Focuses learning on 'hard' examples (faint roots).
    """
    alpha = 0.3
    beta  = 0.7 
    gamma = 4/3
    
    y_true_f = K.flatten(y_true)
    y_pred_f = K.flatten(y_pred)
    
    tp = K.sum(y_true_f * y_pred_f)
    fp = K.sum((1 - y_true_f) * y_pred_f)
    fn = K.sum(y_true_f * (1 - y_pred_f))
    
    tversky = (tp + K.epsilon()) / (tp + alpha * fp + beta * fn + K.epsilon())
    return K.pow((1 - tversky), gamma)

def hybrid_loss(y_true, y_pred):
    """
    Combination of:
    1. Binary Cross Entropy (BCE): Good for global pixel accuracy.
    2. Focal Tversky Loss (FTL): Good for fine structural details.
    """
    bce = tf.keras.losses.binary_crossentropy(y_true, y_pred)
    ftl = focal_tversky_loss(y_true, y_pred)
    
    return bce + ftl

In [ ]:
# 5. MODEL ARCHITECTURE: RESIDUAL ATTENTION U-NET
# A hybrid model combining:
# 1. Residual Connections: To prevent vanishing gradients in deep layers.
# 2. Attention Gates: To focus on roots and suppress background noise in skip connections.

# --- RESIDUAL BLOCK (The "Texture Learner") ---
def residual_block(x, filters, stride=1):
    """
    Standard ResNet block with Identity Shortcut.
    y = f(x) + x
    """
    shortcut = x
    
    # First Conv
    x = Conv2D(filters, (3, 3), strides=stride, padding="same")(x)
    x = BatchNormalization()(x)
    x = Activation("relu")(x)
    
    # Second Conv
    x = Conv2D(filters, (3, 3), padding="same")(x)
    x = BatchNormalization()(x)
    
    # Shortcut handling (if dimensions change due to stride or filter count)
    if stride != 1 or shortcut.shape[-1] != filters:
        shortcut = Conv2D(filters, (1, 1), strides=stride, padding="same")(shortcut)
        shortcut = BatchNormalization()(shortcut)
    
    # Add Residue
    x = Add()([x, shortcut])
    x = Activation("relu")(x)
    return x

# --- ATTENTION BLOCK (The "Noise Filter") ---
def attention_block(x, gating, inter_shape):
    """
    Attention Gate: Uses the 'gating' signal (from decoder) to filter the 'x' signal (from encoder).
    """
    shape_x = K.int_shape(x)
    
    # Align Feature Map (x) to Gating Signal
    theta_x = Conv2D(inter_shape, (2, 2), strides=(1, 1), padding='same')(x)  
    phi_g = Conv2D(inter_shape, (1, 1), padding='same')(gating)
    
    # Add & Activate
    concat_xg = add([theta_x, phi_g])
    act_xg = Activation('relu')(concat_xg)
    
    # Create Attention Map (0 to 1)
    psi = Conv2D(1, (1, 1), padding='same')(act_xg)
    sigmoid_xg = Activation('sigmoid')(psi)
    
    # Upsample Map to match input size
    shape_sigmoid = K.int_shape(sigmoid_xg)
    upsample_sigmoid_xg = UpSampling2D(size=(shape_x[1] // shape_sigmoid[1], 
                                             shape_x[2] // shape_sigmoid[2]))(sigmoid_xg)
    
    # Apply Attention (Element-wise multiplication)
    y = multiply([x, upsample_sigmoid_xg])
    return Conv2D(inter_shape, (1, 1), padding='same')(y)

# --- FULL MODEL ASSEMBLY ---
def res_attention_unet(IMG_HEIGHT, IMG_WIDTH, IMG_CHANNELS):
    inputs = Input((IMG_HEIGHT, IMG_WIDTH, IMG_CHANNELS))
    
    # Initial Conv
    x = Conv2D(32, (3,3), padding='same')(inputs)
    x = BatchNormalization()(x)
    x = Activation('relu')(x)

    # --- ENCODER (Downsampling) ---
    c1 = residual_block(x, 32); p1 = MaxPooling2D((2,2))(c1)
    c2 = residual_block(p1, 64); p2 = MaxPooling2D((2,2))(c2)
    c3 = residual_block(p2, 128); p3 = MaxPooling2D((2,2))(c3)
    c4 = residual_block(p3, 256); p4 = MaxPooling2D((2,2))(c4)
    
    # Bottleneck
    c5 = residual_block(p4, 512)

    # --- DECODER (Upsampling + Attention) ---
    
    # Up 1
    u6 = Conv2DTranspose(256, (2,2), strides=(2,2), padding='same')(c5)
    att6 = attention_block(x=c4, gating=u6, inter_shape=256) # Attention!
    u6 = concatenate([u6, att6])
    c6 = residual_block(u6, 256)
    
    # Up 2
    u7 = Conv2DTranspose(128, (2,2), strides=(2,2), padding='same')(c6)
    att7 = attention_block(x=c3, gating=u7, inter_shape=128) # Attention!
    u7 = concatenate([u7, att7])
    c7 = residual_block(u7, 128)
    
    # Up 3
    u8 = Conv2DTranspose(64, (2,2), strides=(2,2), padding='same')(c7)
    att8 = attention_block(x=c2, gating=u8, inter_shape=64) # Attention!
    u8 = concatenate([u8, att8])
    c8 = residual_block(u8, 64)
    
    # Up 4
    u9 = Conv2DTranspose(32, (2,2), strides=(2,2), padding='same')(c8)
    att9 = attention_block(x=c1, gating=u9, inter_shape=32) # Attention!
    u9 = concatenate([u9, att9])
    c9 = residual_block(u9, 32)

    # Output Layer (Binary Segmentation)
    outputs = Conv2D(1, (1,1), activation='sigmoid')(c9)
    return Model(inputs=[inputs], outputs=[outputs])

In [ ]:
# 6. MODEL COMPILATION
# Build the model
model = res_attention_unet(PATCH_SIZE, PATCH_SIZE, 1)

# Compile with the optimizer, loss, and metrics defined previously
model.compile(
    optimizer=Adam(learning_rate=1e-4),
    loss=hybrid_loss,
    metrics=['accuracy', f1_metric]
)

model.summary()

In [ ]:
# 7. DATA GENERATORS & AUGMENTATION
# Training Generator (Image)
train_datagen = ImageDataGenerator(
    rescale=1./255,             # Normalize inputs to [0,1]
    rotation_range=20,          # Rotation invariance
    horizontal_flip=True,       # Symmetry invariance
    vertical_flip=True,   
    zoom_range=[0.8, 1.2],      # Zooms in/out for scale invariance
    fill_mode='reflect'         # Handle edges naturally
)

# Mask Generator (Must match Image Generator exactly)
train_mask_datagen = ImageDataGenerator(
    rescale=1./255,          
    rotation_range=20,
    horizontal_flip=True,
    vertical_flip=True,
    zoom_range=[0.8, 1.2],      
    fill_mode='reflect'
)

# Validation Generators (Only Rescaling, No Augmentation)
val_datagen = ImageDataGenerator(rescale=1./255)

# (Generators setup - flow from directory)
train_img_gen = train_datagen.flow_from_directory(
    os.path.join(PATCH_DIR, "train_images"),
    target_size=(PATCH_SIZE, PATCH_SIZE),
    class_mode=None, color_mode="grayscale",
    batch_size=BATCH_SIZE, seed=42
)

train_mask_gen = train_mask_datagen.flow_from_directory(
    os.path.join(PATCH_DIR, "train_masks"),
    target_size=(PATCH_SIZE, PATCH_SIZE),
    class_mode=None, color_mode="grayscale",
    batch_size=BATCH_SIZE, seed=42
)

val_img_gen = val_datagen.flow_from_directory(
    os.path.join(PATCH_DIR, "val_images"),
    target_size=(PATCH_SIZE, PATCH_SIZE),
    class_mode=None, color_mode="grayscale",
    batch_size=BATCH_SIZE, seed=42, shuffle=True
)

val_mask_gen = val_datagen.flow_from_directory(
    os.path.join(PATCH_DIR, "val_masks"),
    target_size=(PATCH_SIZE, PATCH_SIZE),
    class_mode=None, color_mode="grayscale",
    batch_size=BATCH_SIZE, seed=42, shuffle=True
)

def combined_generator(img_gen, mask_gen):
    """
    Zips the image and mask generators into a single stream.
    Yields (x_batch, y_batch) tuples required by model.fit()
    """
    while True:
        yield next(img_gen), next(mask_gen)

train_generator = combined_generator(train_img_gen, train_mask_gen)
val_generator   = combined_generator(val_img_gen,   val_mask_gen)

In [ ]:
# 8. TRAINING CALLBACKS
callbacks = [
    # Stop training if validation loss doesn't improve for 12 epochs
    # REQ: restore_best_weights=True ensures we keep the best model, not the last one.
    EarlyStopping(
        monitor='val_loss', 
        patience=12, 
        restore_best_weights=True, 
        verbose=1
    ),
    
    # Reduce learning rate if validation loss stagnates for 4 epochs
    # Helps the model fine-tune its weights when it gets stuck.
    ReduceLROnPlateau(
        monitor='val_loss', 
        factor=0.2, 
        patience=4, 
        min_lr=1e-7, 
        verbose=1
    ),
]

In [ ]:
# 9. MODEL TRAINING
# These logs are the primary proof of model convergence for the reviewer.
history = model.fit(
    train_generator,
    steps_per_epoch=len(train_img_gen),
    validation_data=val_generator,
    validation_steps=len(val_img_gen),
    epochs=50,
    callbacks=callbacks
)

In [ ]:
# Save the best model (restored by EarlyStopping) for inference
MODEL_PATH = f"../models/unet_model_{PATCH_SIZE}px.h5"
os.makedirs(os.path.dirname(MODEL_PATH), exist_ok=True)
model.save(MODEL_PATH)
print(MODEL_PATH)

In [ ]:
best_val_loss = min(history.history['val_loss'])
best_val_f1 = max(history.history['val_f1_metric'])
print(f"Best validation loss: {best_val_loss}")
print(f"Best validation f1: {best_val_f1}")

In [ ]:
loss = history.history['loss']
val_loss = history.history['val_loss']

epochs = np.arange(1, len(loss) + 1)

plt.plot(epochs, loss, label='train_loss')
plt.plot(epochs, val_loss, label='val_loss')
plt.legend()
plt.xlabel('Epoch')
plt.ylabel('Loss (binary_crossentropy)')
_ = plt.xticks(np.arange(1, len(loss) + 1, 3))

In [ ]:
train_f1 = history.history['f1_metric']
val_f1 = history.history['val_f1_metric']

epochs = np.arange(1, len(train_f1) + 1)

plt.plot(epochs, train_f1, label='train_f1')
plt.plot(epochs, val_f1, label='val_f1_metric')
plt.legend()
plt.xlabel('Epoch')
plt.ylabel('F1')
_ = plt.xticks(np.arange(1, len(loss) + 1, 3))